# 📚 Books Data Analysis & Exploratory Data Analysis (EDA)

**Author:** Ranjith B  
**Source:** Scraped from [Books to Scrape](https://books.toscrape.com/) (1,000 titles across all 50 catalog pages)  
**Dataset:** `books_all_pages.csv`  

---

## 🎯 Project Overview & Objectives
This project analyzes 1,000 book records systematically scraped from the web. The objective is to:
1. **Perform comprehensive data cleaning**: fix currency formatting, resolve non-ASCII character encoding artifacts, and convert string star ratings into numerical scales.
2. **Engineer analytical features**: title length, word count, and consumer retail price tiers.
3. **Investigate key business questions**:
   - What is the price distribution of books in the catalog?
   - How are book ratings distributed across the 1-to-5 star scale?
   - Is there a correlation between a book's price and its customer rating?
   - What are the most expensive and most affordable books?

## 1. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set chart styling
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14
})

# Locate raw dataset
data_path = os.path.join('..', 'data', 'raw', 'books_all_pages.csv')
if not os.path.exists(data_path):
    data_path = 'books_all_pages.csv'
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'raw', 'books_all_pages.csv')

df_raw = pd.read_csv(data_path)
print(f"Dataset successfully loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()

## 2. Initial Data Inspection & Health Check

In [ ]:
# Dataset schema and data types
df_raw.info()

In [ ]:
# Check for missing (null) values and duplicates
print("Missing values per column:")
print(df_raw.isnull().sum())
print("\nDuplicate rows:", df_raw.duplicated().sum())

## 3. Data Cleaning & Normalization

Observations from inspection:
1. **`Price`** is stored as text strings with the currency symbol `£` (e.g., `'£51.77'`). We must extract the numeric float value.
2. **`Rating`** is stored as english words (`'One'`, `'Two'`, `'Three'`, `'Four'`, `'Five'`). We map these to integers (1 through 5).
3. **`Title`** contains encoding artifacts (such as UTF-8 bytes read as Windows-1252, e.g., `â€™` instead of apostrophe `’`). We repair these.

In [ ]:
df_clean = df_raw.copy()

# 1. Clean and repair title encoding
def repair_text(text):
    if not isinstance(text, str):
        return text
    try:
        return text.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text.strip()

df_clean['title'] = df_clean['Title'].apply(repair_text).str.strip()

# 2. Extract numeric price in GBP
df_clean['price_gbp'] = df_clean['Price'].astype(str).str.extract(r'(\d+\.\d+)')[0].astype(float)

# 3. Map textual rating to integer 1-5
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df_clean['rating_numeric'] = df_clean['Rating'].map(rating_map)

# 4. Normalize Availability
df_clean['in_stock'] = df_clean['Availability'].str.contains('In stock', case=False, na=False)

# Preview cleaned columns
df_clean[['title', 'price_gbp', 'rating_numeric', 'in_stock']].head()

## 4. Feature Engineering

To enable deeper analysis, we engineer:
- `title_length`: Total character count of the book's title.
- `title_word_count`: Number of words in the title.
- `price_tier`: Categorized into standard retail segments: **Budget (< £20)**, **Mid-Range (£20 - £40)**, and **Premium (> £40)**.

In [ ]:
df_clean['title_length'] = df_clean['title'].str.len()
df_clean['title_word_count'] = df_clean['title'].apply(lambda s: len(s.split()))

price_bins = [0, 20.0, 40.0, float('inf')]
price_labels = ['Budget (< £20)', 'Mid-Range (£20 - £40)', 'Premium (> £40)']
df_clean['price_tier'] = pd.cut(df_clean['price_gbp'], bins=price_bins, labels=price_labels, right=False)

df_clean[['title', 'title_length', 'title_word_count', 'price_tier']].head()

## 5. Statistical Summary of Cleaned Data

In [ ]:
# Statistical summary for all numerical features
df_clean[['price_gbp', 'rating_numeric', 'title_length', 'title_word_count']].describe().round(2)

## 6. Exploratory Data Analysis (EDA)

### 6.1 Univariate Analysis: Book Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

mean_val = df_clean['price_gbp'].mean()
median_val = df_clean['price_gbp'].median()

sns.histplot(df_clean['price_gbp'], kde=True, bins=25, color='#2b5c8f', edgecolor='white', ax=ax)
ax.axvline(mean_val, color='#d9534f', linestyle='--', linewidth=2, label=f'Mean: £{mean_val:.2f}')
ax.axvline(median_val, color='#5cb85c', linestyle='-.', linewidth=2, label=f'Median: £{median_val:.2f}')

ax.set_title('Distribution of Book Prices (£ GBP)', weight='bold')
ax.set_xlabel('Price (£)')
ax.set_ylabel('Number of Books')
ax.legend(frameon=True, facecolor='white')
plt.show()

**Observation:**
- Book prices span evenly from **£10.00 to £59.99** with an average of **£35.07** and a median of **£35.98**.
- The distribution is relatively uniform across the £10–£60 range without heavy clustering, indicating a synthetic catalog design.

### 6.2 Univariate Analysis: Star Ratings Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))

rating_counts = df_clean['rating_numeric'].value_counts().sort_index()
palette = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#1abc9c']

bars = ax.bar(rating_counts.index, rating_counts.values, color=palette, edgecolor='#333333', width=0.6)

for bar in bars:
    h = bar.get_height()
    pct = (h / len(df_clean)) * 100
    ax.annotate(f'{h}\n({pct:.1f}%)', (bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 5), textcoords='offset points', ha='center', weight='bold')

ax.set_title('Catalog Distribution Across Star Ratings (1-5)', weight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Book Count')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1 Star', '2 Stars', '3 Stars', '4 Stars', '5 Stars'])
ax.set_ylim(0, max(rating_counts.values) * 1.18)
plt.show()

**Observation:**
- Ratings are distributed across all 5 tiers: 1 Star (22.6%), 2 Stars (19.6%), 3 Stars (20.3%), 4 Stars (17.9%), and 5 Stars (19.6%).
- There is no customer rating skew; each star tier represents approximately 18% to 23% of the total dataset.

### 6.3 Bivariate Analysis: Does Book Price Vary by Rating?

In [ ]:
# Price summary by rating group
rating_price_stats = df_clean.groupby('rating_numeric')['price_gbp'].agg(
    Book_Count='count',
    Mean_Price='mean',
    Median_Price='median',
    Std_Dev='std',
    Min_Price='min',
    Max_Price='max'
).round(2)
rating_price_stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(x='rating_numeric', y='price_gbp', hue='rating_numeric', data=df_clean,
            palette=palette, ax=ax, width=0.55, linewidth=1.3, legend=False)

means = df_clean.groupby('rating_numeric')['price_gbp'].mean()
ax.plot(range(len(means)), means.values, color='#2c3e50', marker='D', markersize=7,
        linestyle='--', linewidth=1.5, label='Mean Price')

ax.set_title('Price Spread Across Rating Levels', weight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Price (£ GBP)')
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xticklabels(['1 Star', '2 Stars', '3 Stars', '4 Stars', '5 Stars'])
ax.legend(loc='lower right', frameon=True)
plt.show()

**Observation:**
- Mean prices across star tiers are virtually identical: **£34.56** (1 Star) vs **£35.37** (5 Stars).
- Pearson correlation between Price and Rating is **r = 0.028** (near zero), proving that **higher rating does not imply a higher price**.

### 6.4 Market Segmentation: Price Tiers Breakdown

In [ ]:
tier_counts = df_clean['price_tier'].value_counts()[['Budget (< £20)', 'Mid-Range (£20 - £40)', 'Premium (> £40)']]

fig, ax = plt.subplots(figsize=(7, 5.5))
wedges, texts, autotexts = ax.pie(
    tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%',
    startangle=140, colors=['#3498db', '#f39c12', '#9b59b6'],
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_weight('bold')
    t.set_color('white')

ax.set_title('Catalog Breakdown by Price Tier', weight='bold')
plt.show()

### 6.5 Top 5 Most Expensive vs Top 5 Most Affordable Books

In [ ]:
print("=== TOP 5 MOST EXPENSIVE BOOKS ===")
display(df_clean[['title', 'price_gbp', 'rating_numeric']].sort_values(by='price_gbp', ascending=False).head(5))

print("\n=== TOP 5 MOST AFFORDABLE BOOKS ===")
display(df_clean[['title', 'price_gbp', 'rating_numeric']].sort_values(by='price_gbp', ascending=True).head(5))

### 6.6 Feature Correlation Matrix

In [ ]:
corr_matrix = df_clean[['price_gbp', 'rating_numeric', 'title_length', 'title_word_count']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.3f', linewidths=1, ax=ax)
ax.set_title('Feature Correlation Heatmap', weight='bold')
plt.show()

## 7. Key Findings & Conclusions

1. **Uniform Catalog Pricing**: Prices range continuously between £10.00 and £59.99 with a median of £35.98. There are zero pricing outliers.
2. **Independent Star Ratings**: Book price and star rating have no linear relationship ($r = 0.028$). Premium books are no more likely to receive 5 stars than budget books.
3. **Balanced Rating Distribution**: Rating frequencies range evenly between 17.9% and 22.6% per star tier.
4. **Product Segmentation**: 80.4% of the catalog falls into the Mid-Range (£20-£40) and Premium (> £40) tiers, with only 19.6% priced under £20.